# Unit 2 - Data Cleaning and Preparation
## Datasets: GDP per Capita (OECD) x Monthly Unemployment Rate (OECD)

## 0. Setup

In [ ]:
import pandas as pd

GDP_PATH = 'datasets/OECD.SDD.NAD,DSD_NAMAIN1@DF_QNA_EXPENDITURE_CAPITA,+Q.............csv'
UNE_PATH = 'datasets/OECD.SDD.TPS,DSD_LFS@DF_IALFS_UNE_M,+..._Z.Y._T.Y_GE15..M.csv'

AGGREGATE_AREAS = {'EA20', 'EU27_2020', 'G7', 'OECD', 'OECDE', 'USMCA'}

gdp_raw = pd.read_csv(GDP_PATH, low_memory=False)
une_raw = pd.read_csv(UNE_PATH, low_memory=False)

print('GDP raw shape :', gdp_raw.shape)
print('UNE raw shape :', une_raw.shape)

## 1. Duplicate Detection

In [ ]:
# 1a. Full-row exact duplicate check
print('Exact duplicates — GDP:', gdp_raw.duplicated().sum(), '| UNE:', une_raw.duplicated().sum())

# 1b. Partial duplicate check on analytical key
gdp_key_dup = gdp_raw.duplicated(subset=['REF_AREA', 'TIME_PERIOD']).sum()
une_key_dup = une_raw.duplicated(subset=['REF_AREA', 'TIME_PERIOD']).sum()
print('Key-level duplicates (REF_AREA + TIME_PERIOD) — GDP:', gdp_key_dup, '| UNE:', une_key_dup)

# Why does GDP have 226 key duplicates?
mask = gdp_raw.duplicated(subset=['REF_AREA', 'TIME_PERIOD'], keep=False)
print('\nSample: two PRICE_BASE variants per country-period:')
print(gdp_raw[mask][['REF_AREA', 'TIME_PERIOD', 'PRICE_BASE', 'OBS_VALUE']].head(6).to_string())
print('\n-> Structural variants (V=current, LR=constant). Treatment: keep LR only.')

## 2. Column Selection

In [ ]:
GDP_KEEP = ['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE', 'PRICE_BASE', 'OBS_STATUS', 'REF_YEAR_PRICE']
UNE_KEEP = ['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_STATUS']

gdp_clean = gdp_raw[GDP_KEEP].copy()
une_clean = une_raw[UNE_KEEP].copy()

print('GDP after column selection:', gdp_clean.shape)
print('UNE after column selection:', une_clean.shape)
print('\nGDP missing values:')
print(gdp_clean.isnull().sum())
print('\nUNE missing values:')
print(une_clean.isnull().sum())

## 3. Missing Data Treatment
### 3.1 Filter to PRICE_BASE=LR and remove aggregates

In [ ]:
# 3.1 Filter to constant prices (resolves 226 structural duplicates + 50% missing REF_YEAR_PRICE)
print('PRICE_BASE distribution:', gdp_clean['PRICE_BASE'].value_counts().to_dict())
gdp_clean = gdp_clean[gdp_clean['PRICE_BASE'] == 'LR'].drop(columns=['PRICE_BASE'])
print('After PRICE_BASE=LR filter:', gdp_clean.shape)

# 3.2 Remove aggregate areas (not individual countries)
agg_found = sorted(gdp_clean[gdp_clean['REF_AREA'].isin(AGGREGATE_AREAS)]['REF_AREA'].unique())
print('Aggregate areas removed:', agg_found)
gdp_clean = gdp_clean[~gdp_clean['REF_AREA'].isin(AGGREGATE_AREAS)]
print('After removing aggregates:', gdp_clean.shape)
print('\nRemaining missing values:', gdp_clean.isnull().sum().to_dict())
print('OBS_STATUS distribution:', gdp_clean['OBS_STATUS'].value_counts().to_dict())

### 3.2 Parse and align time periods
GDP quarterly (YYYY-QN) vs unemployment monthly (YYYY-MM). Aggregate monthly to quarterly mean.

In [ ]:
# GDP: parse YYYY-QN -> quarter string
gdp_clean['QUARTER'] = pd.PeriodIndex(
    gdp_clean['TIME_PERIOD'].str.replace('-Q', 'Q'), freq='Q'
).astype(str)
gdp_clean = gdp_clean.rename(columns={'OBS_VALUE': 'GDP_PER_CAPITA_USD_PPP'})

# UNE: aggregate monthly -> quarterly mean
une_clean['QUARTER'] = pd.to_datetime(une_clean['TIME_PERIOD'], format='%Y-%m').dt.to_period('Q').astype(str)
une_quarterly = (
    une_clean
    .groupby(['REF_AREA', 'QUARTER'], as_index=False)['OBS_VALUE']
    .mean()
    .rename(columns={'OBS_VALUE': 'UNE_RATE_PCT'})
)

print('GDP sample (quarterly, constant prices):')
print(gdp_clean[['REF_AREA', 'QUARTER', 'GDP_PER_CAPITA_USD_PPP', 'OBS_STATUS']].head(6).to_string())
print('\nUnemployment sample (monthly -> quarterly mean):')
print(une_quarterly.head(6).to_string())

## 4. Merge and Export

In [ ]:
gdp_export = gdp_clean[['REF_AREA', 'QUARTER', 'GDP_PER_CAPITA_USD_PPP', 'OBS_STATUS', 'REF_YEAR_PRICE']].copy()
une_export = une_quarterly.copy()

# Inner join on REF_AREA + QUARTER
merged = pd.merge(gdp_export, une_export, on=['REF_AREA', 'QUARTER'], how='inner')

print('GDP cleaned    :', gdp_export.shape, '  countries:', gdp_export['REF_AREA'].nunique())
print('UNE cleaned    :', une_export.shape, '  countries:', une_export['REF_AREA'].nunique())
print('Merged (final) :', merged.shape, '  countries:', merged['REF_AREA'].nunique(), '  quarters:', merged['QUARTER'].nunique())
print('Quarters       :', sorted(merged['QUARTER'].unique()))
print('Missing values :', merged.isnull().sum().to_dict())
print()
print(merged.sort_values(['REF_AREA', 'QUARTER']).head(12).to_string())

gdp_export.to_csv('datasets/gdp_per_capita_cleaned.csv', index=False)
une_export.to_csv('datasets/unemployment_quarterly_cleaned.csv', index=False)
merged.to_csv('datasets/gdp_unemployment_merged.csv', index=False)
print('\nAll files saved.')